# Topic: Secure Coding Principles - Input validation, whitelist vs blacklist strategies, and boundary checks
 
## 1. THE DEFENSIVE MINDSET
- **Least Privilege**: Components must run with only the minimum access rights needed to do their jobs.
- **Fail-Safe Defaults**: Access decisions should fail closed (deny access by default; explicitly allow verified inputs).
- **Defense in Depth**: Do not rely on a single layer of security. If input validation fails, database paramaterization or container locks should block exploitation.
 
## 2. WHITELIST VS BLACKLIST INPUT FILTERING
- **Blacklisting (Anti-Pattern)**: Trying to detect and block known malicious payloads (e.g. blocking `"<script>"` or `' OR 1=1`).
  - *Vulnerability*: Attackers constantly find evasion techniques (like URL encoding, casing swaps `sCrIpt`, or null byte injection) that bypass filter lists.
- **Whitelisting (Best Practice)**: Defining a strict rule of what is *acceptable* and rejecting everything else.
  - *Example*: Validating a username by checking if it matches `^[a-zA-Z0-9_]{3,16}$`. Any input containing quote symbols or tags is instantly rejected.
 
---


In [ ]:
import re

class UserInputValidator:
    """Validator using strict Whitelisting patterns."""
    
    # Username: alphanumeric and underscore, 3 to 15 characters
    USERNAME_PATTERN = re.compile(r"^[a-zA-Z0-9_]{3,15}$")
    # Email: standard validation
    EMAIL_PATTERN = re.compile(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$")

    @classmethod
    def validate_username(cls, username):
        """Strict whitelist validation with length boundary checks."""
        if not isinstance(username, str):
            return False
            
        # 1. Boundary check: Length constraints
        if len(username) < 3 or len(username) > 15:
            return False
            
        # 2. Pattern check: Whitelist match
        return bool(cls.USERNAME_PATTERN.match(username))

    @classmethod
    def validate_email(cls, email):
        """Strict whitelist email validation."""
        if not isinstance(email, str) or len(email) > 254:
            return False
        return bool(cls.EMAIL_PATTERN.match(email))

print("--- Testing Whitelist Username Validation ---")
test_usernames = [
    "admin_dev",           # Valid
    "ad",                  # Too short
    "admin_with_long_name",# Too long (> 15)
    "admin; DROP TABLE;",  # Injection attempt (contains spaces/semicolons)
    "admin<script>"        # XSS injection attempt
]

for u in test_usernames:
    is_valid = UserInputValidator.validate_username(u)
    print(f"Username: {u:<22} | Is Valid? {is_valid}")
# Expected: Only "admin_dev" is True. All others are blocked.



In [ ]:
print("\n--- Testing Whitelist Email Validation ---")
test_emails = [
    "alice@example.com",
    "invalid-email-no-at.com",
    "malicious_email@site.com;inject_here"
]

for e in test_emails:
    is_valid = UserInputValidator.validate_email(e)
    print(f"Email: {e:<36} | Is Valid? {is_valid}")
